|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>Incidents<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: the incident file<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

You finished Part 3. The KV cache now lives in blocks, a page table maps each
sequence to its blocks, a kernel reads through that table, and blocks can be
shared. Each of these gives a new way to fail. Most of them do not crash.

Each ticket gives you a **symptom** and some **evidence**. Some of the
evidence is noise. Write four lines for each ticket:

1. **Root cause.** One sentence.
2. **The number that proves it.** Not "it looks like". A computation.
3. **The fix.**
4. **The guard.** A test, an assert or an alert that catches it next time.

Four rules:

- The tickets are **not** in the order of the notebooks.
- At least one ticket is **not a bug**. "Nothing is broken" is a valid answer
  only if a number proves it.
- Write your answer **before** you open the solution.
- Every ticket has a scratch cell.

Do this section after stage 09. This notebook needs no GPU.

**The on-call colleague.** In Claude Code, type `/incident 3.1` (or any
other ticket number) to work a ticket as a conversation. The colleague has
access to the system. Ask for a log, a measurement or an experiment, and it
answers with what the system shows. When you write your four lines, it tells
you which lines are weak, and it asks a question about each one. It does not
tell you the cause until you ask for the solution.

### The reference sheet

| GPU | SMs | Bandwidth | bf16 compute |
|---|---|---|---|
| L40S | 142 | 864 GB/s | 362 TFLOP/s |
| A100 SXM 80GB | 108 | 2,039 GB/s | 312 TFLOP/s |

| Model | Layers | Attention heads | KV heads | head_dim | KV bytes per token |
|---|---|---|---|---|---|
| Qwen3-1.7B | 28 | 16 | 8 | 128 | 114,688 (112 KiB) |

- The block size is 16 tokens, unless the ticket says otherwise.
- A DRAM sector is 32 bytes. A full memory transaction is 128 bytes.
- One kernel launch costs about 5 µs.
- With random lengths, the last block of a sequence is on average half
  empty: `(block_size - 1) / 2` empty slots.

# Ticket 1: the pool that shrinks every night

**Severity:** high, in about a week. **Reported by:** the SRE team.

> We check the free blocks at 04:00 each night, when no request is
> active. The number goes down every day. At this rate the server stops
> admitting requests in 9 days.

**Evidence**

- The pool has 20,000 blocks. The free blocks at 04:00: 20,000 on day 1,
  18,160 on day 2, 16,318 on day 3.
- The requests end in two ways. `finish()` runs when a request ends
  normally. `abort()` runs when a client disconnects or times out:

  ```python
  def finish(self, seq):
      self.allocator.free(seq.block_table)
      self.running.remove(seq)

  def abort(self, seq):
      self.running.remove(seq)
  ```

- About 115 requests end with `abort()` each day. At the moment of the
  abort, a request has about 256 tokens.
- An engineer says: "Memory fragmentation grows over time. We must
  restart every week."

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 2: one request in sixteen

**Severity:** critical. Two users are affected at a time. **Reported
by:** users.

> Some answers turn into nonsense after a few tokens. At the same time,
> another user in the same batch gets strange text.

**Evidence**

- 6.2% of the requests go bad.
- The prompt lengths of 12 bad requests: 48, 160, 32, 96, 64, 240, 128,
  176, 80, 32, 112, 224.
- The code that adds a token to a sequence:

  ```python
  def append_token(seq, token):
      seq.num_tokens += 1
      if seq.num_tokens % BLOCK_SIZE == 0:
          seq.block_table.append(allocator.allocate())
      write_kv(seq, position=seq.num_tokens - 1)
  ```

- The prefill allocates `ceil(prompt_len / 16)` blocks.
- The block tables go to the GPU as one tensor. Short tables are padded
  with 0.
- The bad requests are a little longer than the average request.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 3: the cooking bot that talks about law

**Severity:** critical. **Reported by:** a customer of ChefBot.

> I asked ChefBot for a risotto recipe. It told me that "the parties
> agree to the terms above", and it used the language of a contract.

**Evidence**

- One server hosts two products, LegalBot and ChefBot, with automatic
  prefix caching. Each product has its own system prompt. The two system
  prompts end with the same safety paragraph of 64 tokens.
- The hash of a block:

  ```python
  def block_hash(tokens):
      return hash(tuple(tokens))
  ```

- The prefix cache hit rate is 71%. The team expected about 40%.
- The debug log of one ChefBot request: block 0 miss, block 1 miss, ...,
  block 11 miss, block 12 hit, block 13 hit, block 14 hit, block 15 hit.
- LegalBot traffic doubled that week.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 4: the prefix cache that never hits

**Severity:** medium. It costs money. **Reported by:** the performance
team.

> We enabled prefix caching. Every request starts with the same system
> prompt of 1,200 tokens. The time to the first token did not change.
> The hit rate is 0.0%.

**Evidence**

- The hash is chained, as it must be.
- The average user message has 300 tokens.
- The system prompt is a template. Its first line:

      You are the assistant of Acme. Session: {session_id}. Today is {date}.

- The team doubled the size of the prefix cache. The hit rate stayed at
  0.0%.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 5: the kernel with a great L1 hit rate

**Severity:** low. **Reported by:** the kernel team.

> Our paged attention kernel reaches only 190 GB/s on an L40S, 22% of
> the bandwidth. The L1 hit rate is 91%, so the memory is fine. The
> `exp()` in the softmax must be the problem.

**Evidence**

- The kernel gives one thread to one position. Each thread walks the 128
  values of its key row, one bf16 value at a time.
- The [Nsight Compute](../../GLOSSARY.md#nsight-compute) counters:

  | counter | value |
  |---|---|
  | DRAM throughput | 190 GB/s |
  | L1 hit rate | 91% |
  | bytes used for each 32-byte sector | 6.3% |
  | compute pipe utilization | 4% |

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 6: the kernel that is slow only for one user

**Severity:** medium. Every interactive request is batch 1. **Reported
by:** the kernel team.

> At batch 64 our decode attention reaches 610 GB/s on an L40S, 71% of
> the bandwidth. At batch 1 with 4,096 tokens of context it reaches
> 70 GB/s, 8%. The batch-1 path must have a bug.

**Evidence**

- The kernel is stage 08b: one thread block for each (sequence, query
  head). Qwen3-1.7B has 16 query heads.
- The L40S has 142 SMs.
- A colleague says: "At batch 1 the launch overhead dominates."
- The batch-1 path and the batch-64 path run the same kernel.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 7: the faster kernel that holds fewer users

**Severity:** medium. **Reported by:** the capacity team.

> The kernel team changed the block size from 16 to 256, because the
> kernel is 9% faster with large blocks. Since then, the server holds
> 23% fewer concurrent requests before the pool is full.

**Evidence**

- The average sequence has about 400 tokens, and the lengths are
  spread out.
- The pool has the same number of bytes as before.
- The kernel team says: "A block size cannot change the number of
  tokens. The capacity team must have changed something else."

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 8: three of four samples are strange

**Severity:** high. **Reported by:** the team of the writing assistant.

> With `n=4`, the API returns four answers for one prompt. Often three of
> the four are incoherent, and the last one is fine. With `n=1` every
> answer is fine.

**Evidence**

- The four samples share the blocks of the prompt. The reference count
  of those blocks is 4.
- The write of a new token does not check the reference count of its
  block.
- The bug never happens when the prompt length is a multiple of 16.
- The team suspects that the four samples share one random seed.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

### Before you open the solution

Go back to each ticket and write one more line: **which piece of evidence was
noise, and why did it look relevant?**